# 第 6 课：词典 L、消歧符号与 HCLG / CTC-TLG

这一课把第 5 课的真实 `G.fst` 接到发音词典 `L`。你会亲手看到 `#0`、`#1`、`L∘G`，并分清传统 HMM 的 `HCLG` 与 CTC 的 `TLG`。

完成后你应该能够：

1. 说清 `L` 的输入为什么是 phone/token、输出为什么是 word；
2. 区分词典消歧 `#1` 与 LM 回退 `#0`；
3. 用真实 OpenFst 构造并检查 `L∘G`；
4. 从一条最短路径同时读出输入 token、输出词和 LM 代价；
5. 画出 `H∘C∘L∘G` 与 `T∘L∘G` 的层次。


## 0. 先把每一层的接口说清楚

| 图 | 输入标签 | 输出标签 | 主要职责 |
|---|---|---|---|
| `H` | HMM transition-id | context-dependent phone | HMM 拓扑与声学状态 |
| `C` | context-dependent phone | phone | 上下文相关音素展开 |
| `T`（CTC） | frame-level CTC label | collapsed token | blank/repeat 折叠 |
| `L` | phone/token | word | 发音或 token 到词 |
| `G` | word | word/epsilon | N-gram 约束与代价 |

组合的硬条件是：**左图输出标签空间必须与右图输入标签空间一致**。所以 `L` 的输出 word 才能接 `G` 的输入 word。


In [1]:
from pathlib import Path
from math import log
import subprocess
import ipywidgets as widgets
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('请从 learn_asr 项目或 notebooks 目录启动 Jupyter')

ROOT = find_project_root()
PREV = ROOT / 'openfst_lab' / 'lesson05'
LAB = ROOT / 'openfst_lab' / 'lesson06'
LAB.mkdir(parents=True, exist_ok=True)

def run_wsl(*args, check=True):
    result = subprocess.run(
        ['wsl', '-d', 'Ubuntu', '--', *map(str, args)],
        text=True, capture_output=True, check=False, encoding='utf-8', errors='replace',
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'命令失败：{args}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}')
    return result

def to_wsl_path(path):
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(':').lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f'/mnt/{drive}/{relative}'

def write_lf(path, text):
    Path(path).write_text(text, encoding='utf-8', newline='\n')

for command in ['fstcompile', 'fstprint', 'fstinfo', 'fstarcsort', 'fstcompose', 'fstshortestpath']:
    location = run_wsl('which', command).stdout.strip()
    print(f'{command:16s}', location or 'MISSING')
    assert location


fstcompile       /usr/bin/fstcompile
fstprint         /usr/bin/fstprint
fstinfo          /usr/bin/fstinfo


fstarcsort       /usr/bin/fstarcsort
fstcompose       /usr/bin/fstcompose


fstshortestpath  /usr/bin/fstshortestpath


## 1. 读取上一课的真实 G

本课按顺序依赖第 5 课生成的 `words.txt`、`G.kaldi.fst` 和 ARPA。若下面报缺文件，请先完整运行第 5 课。`G.kaldi.fst` 的回退弧输入标签是 `#0`、输出是 epsilon。


In [2]:
words_path = PREV / 'words.txt'
g_path = PREV / 'G.kaldi.fst'
arpa_path = PREV / 'tiny.3gram.arpa'
required = [words_path, g_path, arpa_path]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('请先运行第 5 课，缺少：' + ', '.join(missing))

word_to_id = {}
for line in words_path.read_text(encoding='utf-8').splitlines():
    symbol, index = line.rsplit(maxsplit=1)
    word_to_id[symbol] = int(index)

g_info = run_wsl('fstinfo', to_wsl_path(g_path)).stdout
for line in g_info.splitlines():
    if line.strip().startswith(('# of states', '# of arcs', '# of input epsilons', '# of output epsilons', 'acceptor')):
        print(line.strip())
assert '#0' in word_to_id


# of states                                       71
# of arcs                                         190
# of input epsilons                               0
# of output epsilons                              70
acceptor                                          n


## 2. 一个可读的拼音 token 词典

这里把汉语拼音拆成教学 token。例如 `jintian → jin tian`，`yuyin → yu yin`。这不是生产级发音词典，但输入/输出接口与真实 `L` 相同。

注意两个前缀冲突：

- `hen` 是 `hen hao` 的完整前缀；
- `wo` 是 `wo men` 的完整前缀。

我们在短词结尾附加词典消歧 token `#1`，使两条发音路径可以被明确区分。


In [3]:
LEXICON = {
    '<unk>': ['spn'],
    'bangzhu': ['bang', 'zhu'], 'bucuo': ['bu', 'cuo'],
    'hen': ['hen'], 'henhao': ['hen', 'hao'],
    'jintian': ['jin', 'tian'], 'keyi': ['ke', 'yi'], 'leng': ['leng'],
    'mingtian': ['ming', 'tian'], 'moxing': ['mo', 'xing'],
    'ngram': ['n', 'gram'], 'ni': ['ni'], 'openfst': ['open', 'fst'],
    're': ['re'], 'shibie': ['shi', 'bie'], 'tianqi': ['tian', 'qi'],
    'wo': ['wo'], 'women': ['wo', 'men'], 'xihuan': ['xi', 'huan'],
    'xinqing': ['xin', 'qing'], 'xuexi': ['xue', 'xi'], 'xuyao': ['xu', 'yao'],
    'yuyan': ['yu', 'yan'], 'yuyin': ['yu', 'yin'], 'ziran': ['zi', 'ran'],
    'zuotian': ['zuo', 'tian'],
}
lm_words = set(word_to_id) - {'<eps>', '<s>', '</s>', '#0'}
assert lm_words == set(LEXICON), (lm_words - set(LEXICON), set(LEXICON) - lm_words)

prefix_words = {
    short for short, short_pron in LEXICON.items()
    if any(short != other and other_pron[:len(short_pron)] == short_pron
           for other, other_pron in LEXICON.items())
}
print('需要 #1 的前缀词：', sorted(prefix_words))
assert prefix_words == {'hen', 'wo'}


需要 #1 的前缀词： ['hen', 'wo']


### `#1` 与 `#0` 不是同一个东西

- `#1`：属于 **L 的输入侧**，解决发音相同或前缀发音带来的词典歧义；
- `#0`：把 **L 的输出侧** 与 **G 的回退输入侧**连接起来。`G` 走低阶 N-gram 时要经过它；
- 它们最终通常会在完整建图流程中被特殊处理或变成 epsilon，不会作为识别文本显示给用户。


In [4]:
phone_symbols = sorted({token for pron in LEXICON.values() for token in pron})
phones = ['<eps>', *phone_symbols, '#0', '#1']
phones_path = LAB / 'phones.txt'
write_lf(phones_path, '\n'.join(f'{symbol} {index}' for index, symbol in enumerate(phones)) + '\n')
phone_to_id = {symbol: index for index, symbol in enumerate(phones)}
print(phones_path.read_text(encoding='utf-8'))
assert phone_to_id['<eps>'] == 0


<eps> 0
bang 1
bie 2
bu 3
cuo 4
fst 5
gram 6
hao 7
hen 8
huan 9
jin 10
ke 11
leng 12
men 13
ming 14
mo 15
n 16
ni 17
open 18
qi 19
qing 20
ran 21
re 22
shi 23
spn 24
tian 25
wo 26
xi 27
xin 28
xing 29
xu 30
xue 31
yan 32
yao 33
yi 34
yin 35
yu 36
zhu 37
zi 38
zuo 39
#0 40
#1 41



## 3. 构造 L.fst

每个词的第一条弧输出 word，后续 token 输出 epsilon；一个词读完后回到状态 0，因此可以继续识别下一个词。状态 0 既是开始状态也是终止状态。

额外加入 `#0:#0` 自环：左边是 phone 表中的 `#0`，右边是 word 表中的 `#0`。虽然字符串相同，两个整数 ID 不要求相同。


In [5]:
def build_lexicon_text(include_backoff_loop=True):
    lines = []
    next_state = 1
    for word in sorted(LEXICON):
        tokens = [*LEXICON[word], *(['#1'] if word in prefix_words else [])]
        source = 0
        for index, token in enumerate(tokens):
            is_last = index == len(tokens) - 1
            destination = 0 if is_last else next_state
            if not is_last:
                next_state += 1
            output = word if index == 0 else '<eps>'
            lines.append(f'{source} {destination} {token} {output} 0')
            source = destination
    if include_backoff_loop:
        lines.append('0 0 #0 #0 0')
    lines.append('0')
    return '\n'.join(lines) + '\n'

l_text = LAB / 'L.txt'
l_fst = LAB / 'L.fst'
write_lf(l_text, build_lexicon_text())
run_wsl(
    'fstcompile', f'--isymbols={to_wsl_path(phones_path)}', f'--osymbols={to_wsl_path(words_path)}',
    '--keep_isymbols=true', '--keep_osymbols=true',
    to_wsl_path(l_text), to_wsl_path(l_fst),
)
printed_l = run_wsl('fstprint', to_wsl_path(l_fst)).stdout
print('L 的前 18 行：')
print('\n'.join(printed_l.splitlines()[:18]))
assert any('#0' in line and line.split()[3] == '#0' for line in printed_l.splitlines())
assert any('#1' in line for line in printed_l.splitlines())


L 的前 18 行：
0	0	spn	<unk>
0	1	bang	bangzhu
0	2	bu	bucuo
0	3	hen	hen
0	4	hen	henhao
0	5	jin	jintian
0	6	ke	keyi
0	0	leng	leng
0	7	ming	mingtian
0	8	mo	moxing
0	9	n	ngram
0	0	ni	ni
0	10	open	openfst
0	0	re	re
0	11	shi	shibie
0	12	tian	tianqi
0	13	wo	wo
0	14	wo	women


## 4. 排序并组合 L∘G

`fstcompose L G` 匹配的是 `L.olabel == G.ilabel`。为了让匹配高效，左图按 output label 排序，右图按 input label 排序。符号表路径一致还不够，真正重要的是每个整数 ID 的含义一致。


In [6]:
l_sorted = LAB / 'L.osorted.fst'
g_sorted = LAB / 'G.isorted.fst'
lg_fst = LAB / 'LG.fst'
run_wsl('fstarcsort', '--sort_type=olabel', to_wsl_path(l_fst), to_wsl_path(l_sorted))
run_wsl('fstarcsort', '--sort_type=ilabel', to_wsl_path(g_path), to_wsl_path(g_sorted))
run_wsl('fstcompose', to_wsl_path(l_sorted), to_wsl_path(g_sorted), to_wsl_path(lg_fst))

info = run_wsl('fstinfo', to_wsl_path(lg_fst)).stdout
selected = [line.strip() for line in info.splitlines() if line.strip().startswith((
    '# of states', '# of arcs', '# of input epsilons', '# of output epsilons',
    'input deterministic', 'output deterministic', 'acceptor'))]
print('L∘G')
print('\n'.join(selected))
state_count = int(next(line.split()[-1] for line in selected if line.startswith('# of states')))
assert state_count > 0


L∘G
# of states                                       133
# of arcs                                         252
# of input epsilons                               0
# of output epsilons                              132
acceptor                                          n
input deterministic                               n
output deterministic                              y


## 5. 用输出词串约束 LG，再取最短路径

我们构造一个只接受指定 word 序列的线性 acceptor `S`，计算 `(L∘G)∘S`。这样既限定了输出句子，又让最短路径自动告诉我们输入侧需要哪些普通 token、`#1` 和 `#0`。


In [7]:
def make_word_acceptor(sentence, stem):
    words = sentence.split()
    unknown = [word for word in words if word not in LEXICON]
    if unknown:
        raise ValueError(f'词典中没有：{unknown}')
    text_path = LAB / f'{stem}.words.txt'
    fst_path = LAB / f'{stem}.words.fst'
    lines = [f'{i} {i+1} {word} 0' for i, word in enumerate(words)]
    lines.append(str(len(words)))
    write_lf(text_path, '\n'.join(lines) + '\n')
    run_wsl(
        'fstcompile', '--acceptor=true',
        f'--isymbols={to_wsl_path(words_path)}', f'--osymbols={to_wsl_path(words_path)}',
        '--keep_isymbols=true', '--keep_osymbols=true',
        to_wsl_path(text_path), to_wsl_path(fst_path),
    )
    return fst_path

def path_weight(printed):
    total = 0.0
    for line in printed.splitlines():
        fields = line.split()
        if len(fields) >= 5:
            total += float(fields[4])
        elif len(fields) == 2:
            total += float(fields[1])
    return total

def best_lg_path(sentence, stem='sentence', graph=lg_fst):
    sentence_fst = make_word_acceptor(sentence, stem)
    sentence_sorted = LAB / f'{stem}.words.isorted.fst'
    graph_sorted = LAB / f'{stem}.graph.osorted.fst'
    constrained = LAB / f'{stem}.constrained.fst'
    best = LAB / f'{stem}.best.fst'
    run_wsl('fstarcsort', '--sort_type=ilabel', to_wsl_path(sentence_fst), to_wsl_path(sentence_sorted))
    run_wsl('fstarcsort', '--sort_type=olabel', to_wsl_path(graph), to_wsl_path(graph_sorted))
    run_wsl('fstcompose', to_wsl_path(graph_sorted), to_wsl_path(sentence_sorted), to_wsl_path(constrained))
    states = int(next(line.split()[-1] for line in run_wsl('fstinfo', to_wsl_path(constrained)).stdout.splitlines()
                      if line.strip().startswith('# of states')))
    if states == 0:
        return None
    run_wsl('fstshortestpath', to_wsl_path(constrained), to_wsl_path(best))
    printed = run_wsl('fstprint', to_wsl_path(best)).stdout
    # fstprint 按状态编号打印，不保证就是从 start 到 final 的阅读顺序；显式沿弧回溯。
    arc_fields = [line.split() for line in printed.splitlines() if len(line.split()) >= 4]
    arcs_by_source = {int(fields[0]): fields for fields in arc_fields}
    best_info = run_wsl('fstinfo', to_wsl_path(best)).stdout
    state = int(next(line.split()[-1] for line in best_info.splitlines() if line.strip().startswith('initial state')))
    ordered_arcs = []
    while state in arcs_by_source:
        fields = arcs_by_source[state]
        ordered_arcs.append(fields)
        state = int(fields[1])
    inputs = [fields[2] for fields in ordered_arcs if fields[2] != '<eps>']
    outputs = [fields[3] for fields in ordered_arcs if fields[3] not in {'<eps>', '#0'}]
    ordered_text = '\n'.join('\t'.join(fields) for fields in ordered_arcs)
    return {'inputs': inputs, 'outputs': outputs, 'cost': path_weight(printed), 'printed': ordered_text}

demo_sentence = 'wo xuexi yuyan moxing'
demo = best_lg_path(demo_sentence, 'demo')
assert demo is not None
print('输出词：', ' '.join(demo['outputs']))
print('输入 token：', ' '.join(demo['inputs']))
print('LM 路径代价：', round(demo['cost'], 6))
print('\n原始最短路径：\n' + demo['printed'])
assert demo['outputs'] == demo_sentence.split()
assert '#1' in demo['inputs'], 'wo 是前缀词，路径中应出现 #1'
assert '#0' in demo['inputs'], '未见过的高阶历史需要通过 #0 回退'


输出词： wo xuexi yuyan moxing
输入 token： wo #1 xue xi #0 yu yan mo xing
LM 路径代价： 7.463854

原始最短路径：
9	8	wo	wo	2.17217803
8	7	#1	<eps>
7	6	xue	xuexi	1.28914499
6	5	xi	<eps>
5	4	#0	<eps>	0.529259384
4	3	yu	yuyan	2.39607358
3	2	yan	<eps>
2	1	mo	moxing	0.51773572
1	0	xing	<eps>


## 6. 再次验证：LG 路径代价等于 ARPA 概率代价

ARPA 用 `log10(P)`；OpenFst tropical 权重是 `-ln(P)`。因此：

$$cost_{FST}=-log_{10}(P_{ARPA})\ln(10)$$

如果 `L` 的词典权重为 0，`L∘G` 的路径代价应全部来自 `G`。


In [8]:
def parse_arpa(path):
    entries = {}
    order = 0
    for raw in Path(path).read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if line.startswith('\\') and line.endswith('-grams:'):
            order = int(line[1:].split('-')[0])
            entries[order] = {}
        elif order and line and not line.startswith('\\'):
            fields = line.split()
            probability = float(fields[0])
            ngram = tuple(fields[1:1 + order])
            backoff = float(fields[1 + order]) if len(fields) > 1 + order else 0.0
            entries[order][ngram] = (probability, backoff)
    return entries

arpa = parse_arpa(arpa_path)
max_order = max(arpa)

def arpa_word_log10(word, history):
    accumulated_backoff = 0.0
    max_history = min(len(history), max_order - 1)
    for history_length in range(max_history, -1, -1):
        current_history = tuple(history[-history_length:]) if history_length else ()
        candidate = current_history + (word,)
        if candidate in arpa[len(candidate)]:
            return accumulated_backoff + arpa[len(candidate)][candidate][0]
        if current_history:
            accumulated_backoff += arpa[len(current_history)].get(current_history, (0.0, 0.0))[1]
    raise KeyError(word)

def sentence_arpa_cost(sentence):
    tokens = ['<s>', *sentence.split(), '</s>']
    total_log10 = 0.0
    for index in range(1, len(tokens)):
        total_log10 += arpa_word_log10(tokens[index], tokens[:index])
    return -total_log10 * log(10)

for index, sentence in enumerate([
    'jintian tianqi henhao',
    'jintian xinqing bucuo',
    'wo xuexi yuyan moxing',
]):
    result = best_lg_path(sentence, f'compare_{index}')
    assert result is not None
    arpa_cost = sentence_arpa_cost(sentence)
    difference = abs(result['cost'] - arpa_cost)
    print(f'{sentence:28s} ARPA={arpa_cost:.6f} LG={result["cost"]:.6f} diff={difference:.2e}')
    assert difference < 1e-4


jintian tianqi henhao        ARPA=4.553500 LG=4.553500 diff=2.98e-08


jintian xinqing bucuo        ARPA=4.715209 LG=4.715209 diff=3.24e-08


wo xuexi yuyan moxing        ARPA=7.463854 LG=7.463854 diff=2.30e-07


## 7. 失败对照：删掉 L 的 `#0:#0` 自环

删掉自环以后，L 无法向 G 提供回退标签。训练中未见过的高阶上下文需要回退时，路径会消失。下面只改变这一处，其余完全相同。


In [9]:
l_bad_text = LAB / 'L.no_backoff_loop.txt'
l_bad = LAB / 'L.no_backoff_loop.fst'
l_bad_sorted = LAB / 'L.no_backoff_loop.osorted.fst'
lg_bad = LAB / 'LG.no_backoff_loop.fst'
write_lf(l_bad_text, build_lexicon_text(include_backoff_loop=False))
run_wsl(
    'fstcompile', f'--isymbols={to_wsl_path(phones_path)}', f'--osymbols={to_wsl_path(words_path)}',
    '--keep_isymbols=true', '--keep_osymbols=true',
    to_wsl_path(l_bad_text), to_wsl_path(l_bad),
)
run_wsl('fstarcsort', '--sort_type=olabel', to_wsl_path(l_bad), to_wsl_path(l_bad_sorted))
run_wsl('fstcompose', to_wsl_path(l_bad_sorted), to_wsl_path(g_sorted), to_wsl_path(lg_bad))
bad_result = best_lg_path(demo_sentence, 'bad_demo', graph=lg_bad)
print('删除 #0:#0 后：', '路径消失' if bad_result is None else bad_result)
assert bad_result is None


删除 #0:#0 后： 路径消失


## 8. 传统 HCLG 与 CTC-TLG

```text
传统 HMM：  transition-id ─H→ context-phone ─C→ phone ─L→ word ─G→ LM path
CTC：       frame label  ─T→ collapsed token ─L→ word ─G→ LM path
```

`HCLG` 是 Kaldi 式 HMM 系统中常见的整体组合名。CTC 没有同样的 HMM transition-id/context-dependency 结构，通常由 CTC topology `T` 处理 blank 和重复，再接 `L`、`G`；不同工具链会叫 `TLG`、token graph 或 decoding graph。名字会变，接口检查方法不变。

CTC 折叠的最小规则：先合并相邻重复，再删除 blank。注意 `a blank a → aa`，而 `a a → a`。


In [10]:
def ctc_collapse(path, blank='<blk>'):
    collapsed = []
    previous = None
    for token in path:
        if token != previous and token != blank:
            collapsed.append(token)
        previous = token
    return collapsed

examples = [
    ['<blk>', 'jin', 'jin', '<blk>', 'tian'],
    ['a', 'a'],
    ['a', '<blk>', 'a'],
]
for path in examples:
    print(path, '→', ctc_collapse(path))
assert ctc_collapse(['a', 'a']) == ['a']
assert ctc_collapse(['a', '<blk>', 'a']) == ['a', 'a']


['<blk>', 'jin', 'jin', '<blk>', 'tian'] → ['jin', 'tian']
['a', 'a'] → ['a']
['a', '<blk>', 'a'] → ['a', 'a']


## 9. 交互检查不同句子的 LG 路径

选择句子后重点看：何时出现 `#1`，何时出现一个或多个 `#0`，以及这些符号为什么没有出现在最终输出词串。


In [11]:
sentence_widget = widgets.Dropdown(
    options=[
        'jintian tianqi henhao',
        'wo xuexi yuyan moxing',
        'wo xihuan yuyin shibie',
        'mingtian women xuexi ngram',
    ],
    description='句子：', layout=widgets.Layout(width='520px'),
)
output_widget = widgets.Output()

def refresh_path(change=None):
    with output_widget:
        output_widget.clear_output(wait=True)
        result = best_lg_path(sentence_widget.value, 'interactive')
        if result is None:
            print('没有路径')
        else:
            print('输入：', ' '.join(result['inputs']))
            print('输出：', ' '.join(result['outputs']))
            print('代价：', round(result['cost'], 6))

sentence_widget.observe(refresh_path, names='value')
display(sentence_widget, output_widget)
refresh_path()


Dropdown(description='句子：', layout=Layout(width='520px'), options=('jintian tianqi henhao', 'wo xuexi yuyan mo…

Output()

## 10. 自动判题（先闭卷填写）

每个答案尽量只写一个关键词。运行后不足 7/8 时，回到对应实验找证据，而不是背答案。


In [12]:
questions = [
    '1. L 的输入标签类型是什么？',
    '2. L 的输出必须匹配 G 的哪一侧？',
    '3. G 的回退标签是什么？',
    '4. 本课用于词典前缀消歧的标签是什么？',
    '5. tropical 代价越高越好还是越低越好？',
    '6. 传统 HMM 完整解码图常写成什么？',
    '7. 本课给出的 CTC 图链路缩写是什么？',
    '8. 为二遍语言模型保留候选，应保存 N-best 还是只保存 1-best？',
]
for question in questions:
    print(question)

answers = ['', '', '', '', '', '', '', '']
expected = [
    {'phone', 'token', 'phone/token'}, {'input', '输入', '输入侧'}, {'#0'}, {'#1'},
    {'低', '越低越好', 'lower'}, {'hclg', 'h∘c∘l∘g'}, {'tlg', 't∘l∘g'},
    {'n-best', 'nbest', 'n best'},
]

def normalize_answer(value):
    return str(value).strip().lower().replace(' ', '')

score = 0
for index, (answer, accepted) in enumerate(zip(answers, expected), start=1):
    accepted_normalized = {normalize_answer(item) for item in accepted}
    correct = normalize_answer(answer) in accepted_normalized
    score += int(correct)
    print(('✅' if correct else '❌'), f'第 {index} 题')
print(f'得分：{score}/8；达到 7/8 再进入下一课。')


1. L 的输入标签类型是什么？
2. L 的输出必须匹配 G 的哪一侧？
3. G 的回退标签是什么？
4. 本课用于词典前缀消歧的标签是什么？
5. tropical 代价越高越好还是越低越好？
6. 传统 HMM 完整解码图常写成什么？
7. 本课给出的 CTC 图链路缩写是什么？
8. 为二遍语言模型保留候选，应保存 N-best 还是只保存 1-best？
❌ 第 1 题
❌ 第 2 题
❌ 第 3 题
❌ 第 4 题
❌ 第 5 题
❌ 第 6 题
❌ 第 7 题
❌ 第 8 题
得分：0/8；达到 7/8 再进入下一课。


## 11. 离场票

在不看上文的情况下完成：

- [ ] 画出 `L` 的输入、输出，并解释 `L.olabel == G.ilabel`；
- [ ] 用自己的话区分 `#1` 和 `#0`；
- [ ] 从 `fstprint` 的一行读出源状态、目标状态、输入、输出和权重；
- [ ] 解释为什么删掉 `#0:#0` 后某些回退句子没有路径；
- [ ] 画出传统 `HCLG` 与 CTC `TLG`，标注每层标签空间；
- [ ] 自动判题至少 7/8。

下一课：[N-best、Lattice、分数融合与二遍重打分](语言模型零基础_07_Nbest_Lattice分数融合与二遍重打分.ipynb)：从候选图加入声学分、LM scale、插词项、热词与二遍重打分。
